In [2]:
import requests
from bs4 import BeautifulSoup
import json
from pathlib import Path

URL = "https://subastas.boe.es/subastas_ava.php"
SEEN_FILE = Path("subastas_madrid_vistas.json")
TOPIC = "subastas-madrid-vivienda-350k"

data = {
    "campo[0]": "SUBASTA.ORIGEN", "dato[0]": "",
    "campo[1]": "SUBASTA.AUTORIDAD", "dato[1]": "",
    "campo[2]": "SUBASTA.ESTADO.CODIGO", "dato[2]": "PU",
    "campo[3]": "BIEN.TIPO", "dato[3]": "I",
    "dato[4]": "501",
    "campo[5]": "BIEN.DIRECCION", "dato[5]": "",
    "campo[6]": "BIEN.CODPOSTAL", "dato[6]": "",
    "campo[7]": "BIEN.LOCALIDAD", "dato[7]": "Madrid",
    "campo[8]": "BIEN.COD_PROVINCIA", "dato[8]": "28",
    "campo[9]": "SUBASTA.POSTURA_MINIMA_MINIMA_LOTES",
    "dato[9]": "35000000",
    "page_hits": "50",
    "sort_field[0]": "SUBASTA.FECHA_FIN",
    "sort_order[0]": "asc",
    "accion": "Buscar"
}

def notify(msg):
    requests.post(
        f"https://ntfy.sh/{TOPIC}",
        data=msg.encode("utf-8"),
        headers={"Title": "Nueva vivienda BOE Madrid"}
    )

seen = set(json.loads(SEEN_FILE.read_text())) if SEEN_FILE.exists() else set()

r = requests.post(URL, data=data, timeout=20)
soup = BeautifulSoup(r.text, "html.parser")

for a in soup.select("a[href*='subasta.php'], a[href*='idSub=']"):
    link = "https://subastas.boe.es/" + a["href"].lstrip("./")
    title = a.get_text(" ", strip=True)

    if "SUB-" in title and link not in seen:
        notify(f"{title}\n{link}")
        seen.add(link)

SEEN_FILE.write_text(json.dumps(list(seen), indent=2))

3423

In [6]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import json
import re
from urllib.parse import urljoin

URL = "https://subastas.boe.es/subastas_ava.php"
BASE = "https://subastas.boe.es/"

# CAMBIA ESTO por el mismo topic que pongas en la app ntfy
NTFY_TOPIC = "subastas-madrid-vivienda-jorge-350k"

SEEN_FILE = Path("subastas_vistas_2.json")

DATA = {
    "campo[0]": "SUBASTA.ORIGEN",
    "dato[0]": "",

    "campo[1]": "SUBASTA.AUTORIDAD",
    "dato[1]": "",

    "campo[2]": "SUBASTA.ESTADO.CODIGO",
    "dato[2]": "PU",  # Próxima apertura

    "campo[3]": "BIEN.TIPO",
    "dato[3]": "I",  # Inmuebles

    "campo[4]": "BIEN.SUBTIPO",
    "dato[4]": "501",  # Vivienda

    "campo[5]": "BIEN.DIRECCION",
    "dato[5]": "",

    "campo[6]": "BIEN.CODPOSTAL",
    "dato[6]": "",

    "campo[7]": "BIEN.LOCALIDAD",
    "dato[7]": "Madrid",

    "campo[8]": "BIEN.COD_PROVINCIA",
    "dato[8]": "28",  # Madrid

    "campo[9]": "SUBASTA.POSTURA_MINIMA_MINIMA_LOTES",
    "dato[9]": "35000000",  # 350.000 € en céntimos

    "campo[10]": "SUBASTA.NUM_CUENTA_EXPEDIENTE_1",
    "dato[10]": "",

    "campo[11]": "SUBASTA.NUM_CUENTA_EXPEDIENTE_2",
    "dato[11]": "",

    "campo[12]": "SUBASTA.NUM_CUENTA_EXPEDIENTE_3",
    "dato[12]": "",

    "campo[13]": "SUBASTA.NUM_CUENTA_EXPEDIENTE_4",
    "dato[13]": "",

    "campo[14]": "SUBASTA.NUM_CUENTA_EXPEDIENTE_5",
    "dato[14]": "",

    "campo[15]": "SUBASTA.ID_SUBASTA_BUSCAR",
    "dato[15]": "",

    "campo[16]": "SUBASTA.ACREEDORES",
    "dato[16]": "",

    "campo[17]": "SUBASTA.FECHA_FIN",
    "dato[17][0]": "",
    "dato[17][1]": "",

    "campo[18]": "SUBASTA.FECHA_INICIO",
    "dato[18][0]": "",
    "dato[18][1]": "",

    "page_hits": "50",
    "sort_field[0]": "SUBASTA.FECHA_FIN",
    "sort_order[0]": "asc",
    "accion": "Buscar",
}


def cargar_vistas():
    if SEEN_FILE.exists():
        return set(json.loads(SEEN_FILE.read_text(encoding="utf-8")))
    return set()


def guardar_vistas(vistas):
    SEEN_FILE.write_text(
        json.dumps(sorted(list(vistas)), indent=2, ensure_ascii=False),
        encoding="utf-8"
    )


def notificar(mensaje):
    url = f"https://ntfy.sh/{NTFY_TOPIC}"
    requests.post(
        url,
        data=mensaje.encode("utf-8"),
        headers={
            "Title": "Nueva subasta vivienda Madrid",
            "Priority": "high",
            "Tags": "house,warning"
        },
        timeout=20
    )


def buscar_subastas():
    r = requests.post(URL, data=DATA, timeout=30)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    subastas = {}

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if "detalleSubasta.php?idSub=" not in href:
            continue

        match = re.search(r"idSub=([^&]+)", href)
        if not match:
            continue

        id_subasta = match.group(1)
        link = urljoin(BASE, href)

        subastas[id_subasta] = link

    return subastas


def main():
    vistas = cargar_vistas()
    subastas = buscar_subastas()

    nuevas = 0

    for id_subasta, link in subastas.items():
        if id_subasta not in vistas:
            mensaje = (
                f"Nueva subasta de vivienda en Madrid\n\n"
                f"{id_subasta}\n"
                f"Postura mínima inferior a 350.000 €\n\n"
                f"{link}"
            )

            notificar(mensaje)
            vistas.add(id_subasta)
            nuevas += 1

    guardar_vistas(vistas)

    print(f"Subastas encontradas: {len(subastas)}")
    print(f"Nuevas notificadas: {nuevas}")


if __name__ == "__main__":
    main()

Subastas encontradas: 5
Nuevas notificadas: 5
